# Model Interpretation....

This last part of my analysis will be focussing on addressing the issues and limitation of the model from part 5. We saw earlier analyzing the Bias-Variance trade-off that lower review count bucket sizes show that train and test have a large gap within our Logistic Regression and our BILSTM model. This was a important discovery because it revealed to us that our model's limitation is caused by the lack of data coverage for certain grape types. However, the counter argument is that there is sufficient data to train the model and that the cause of this is due to the quality of our data is not addressing all of the variation within a grape type. 

Thus, in this notebook we are going to expand on our surface level understanding that our model provided us to see if the limitation is caused by the lack of data or the quality of description within our data. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support

model_df = pd.read_pickle("model_data.pkl")
classes = pd.read_csv("label_classes.csv")["grape"].to_numpy()

proba_test_ens = np.load("proba_test_ens.npy")
proba_test_m1 = np.load("proba_test_m1.npy")
proba_test_m2 = np.load("proba_test_m2.npy")


In [2]:
test = model_df[model_df["split"] == "test"].reset_index(drop=True)
assert len(test) == len(proba_test_ens), "test rows don't line up with the saved predictions"

test["pred"] = classes[proba_test_ens.argmax(axis=1)]
test["confidence"] = proba_test_ens.max(axis=1)
test["correct"] = test["pred"] == test["grape"]
test["name_in_text"] = test["description_masked"].str.contains("varietyname")


In [3]:
prec, rec, f1, support = precision_recall_fscore_support(
    test["grape"], test["pred"], labels=classes, zero_division=0
)
train_counts = model_df.loc[model_df["split"] == "train", "grape"].value_counts()

scorecard = pd.DataFrame({
    "grape": classes,
    "precision": prec, 
    "recall": rec,
    "f1": f1, 
    "test_n": support
}).set_index("grape")
scorecard["train_n"] = train_counts
scorecard = scorecard.sort_values("f1")

bottom3 = scorecard.index[:3].tolist() # bottom 3
top3 = scorecard.index[-3:].tolist() # top 3
print("Bottom 3:", bottom3)
print("top 3:", top3)
scorecard.round(3)

Bottom 3: ['Pinot Blanc', 'Grenache', 'Cabernet Franc']
top 3: ['Chardonnay', 'Nebbiolo', 'Glera']


,precision,recall,f1,test_n,train_n
grape,,,,,
Pinot Blanc,0.592,0.356,0.444,118,554
Grenache,0.609,0.370,0.461,189,882
Cabernet Franc,0.676,0.398,0.501,251,1169
Petite Sirah,0.634,0.470,0.540,151,707
Malbec,0.603,0.538,0.569,517,2412
Merlot,0.614,0.543,0.576,713,3330
Tempranillo,0.537,0.628,0.579,444,2071
Chenin Blanc,0.653,0.525,0.582,122,566
Carmenère,0.567,0.600,0.583,120,559


In [4]:
bottom_df = test[test["grape"].isin(bottom3)].copy()
top_df = test[test["grape"].isin(top3)].copy()

for name, frame in [("Bottom 3", bottom_df), ("Top 3", top_df)]:
    print(f"{name}: {len(frame)} test reviews | accuracy {frame['correct'].mean():.3f} "
          f"| name in text {frame['name_in_text'].mean():.1%}")

Bottom 3: 558 test reviews | accuracy 0.380 | name in text 38.5%
Top 3: 3007 test reviews | accuracy 0.880 | name in text 33.1%


# First Observation
Comparing basic metrics between the 3 best- and 3 worst-classified grapes, about 30–40% of reviews in both groups mention the grape by name. Surprisingly, the bottom 3 mention their grape name more often than the top 3.

We expected the opposite. In Part 5, our model scored much higher on reviews that named the grape than on reviews that didn't (macro F1 0.921 vs 0.536), so the name is a strong clue. If the name were what separates easy grapes from hard ones, the bottom 3 should perform better. Instead, they perform worse.

So the grape name doesn't explain why the bottom 3 are hard to classify. The difference must come from the rest of the description. Next, we test which part: tasting language (fruit and flavor words), structure words (tannin, acidity, body), or place names, which our masking leaves in.

# Exploring Our Bottom Three

In [5]:
bottom_df[bottom_df["grape"] == "Pinot Blanc"].head(5)

,review_id,variety,grape,bucket,split,country,province,taster_name,description,description_masked,emb_row,y,own_name_in_text,any_name_in_text,pred,confidence,correct,name_in_text
127,1373,Pinot Bianco,Pinot Blanc,500-999,test,Italy,Northeastern Italy,NaN,This is a very fragrant and pungent Pinot Bian...,This is a very fragrant and pungent varietynam...,1372,17,True,True,Pinot Blanc,0.574403,True,True
214,2418,Pinot Blanc,Pinot Blanc,500-999,test,US,Oregon,Paul Gregutt,"Tart and dry, this has a broad flavor that doe...","Tart and dry, this has a broad flavor that doe...",2417,17,False,False,Merlot,0.162156,False,False
244,2714,Pinot Blanc,Pinot Blanc,500-999,test,France,Alsace,Anne Krebiehl MW,Friendly apple fruit sits easily on nose and p...,Friendly apple fruit sits easily on nose and p...,2713,17,False,False,Grüner Veltliner,0.314075,False,False
247,2733,Pinot Blanc,Pinot Blanc,500-999,test,France,Alsace,Anne Krebiehl MW,"This is a very simple, friendly, fresh and liv...","This is a very simple, friendly, fresh and liv...",2732,17,False,False,Grüner Veltliner,0.306997,False,False
521,5475,Weissburgunder,Pinot Blanc,500-999,test,Austria,Burgenland,Anne Krebiehl MW,Very subtle hints of green and yellow pears pl...,Very subtle hints of green and yellow pears pl...,5474,17,True,True,Pinot Blanc,0.561022,True,True


In [6]:
bottom_df[bottom_df["grape"] == "Grenache"].head(5)

,review_id,variety,grape,bucket,split,country,province,taster_name,description,description_masked,emb_row,y,own_name_in_text,any_name_in_text,pred,confidence,correct,name_in_text
44,522,Grenache,Grenache,1000-1999,test,US,California,Matt Kettmann,This is a delicate yet flavorful take on this ...,This is a delicate yet flavorful take on this ...,521,10,False,False,Grenache,0.582888,True,False
250,2751,Garnacha,Grenache,1000-1999,test,US,California,Jim Gordon,"This wine smells fruity, like raspberry and cr...","This wine smells fruity, like raspberry and cr...",2750,10,False,False,Pinot Noir,0.289108,False,False
305,3427,Grenache,Grenache,1000-1999,test,US,Washington,Sean P. Sullivan,Aromas of vanilla and cocoa mix with red fruit...,Aromas of vanilla and cocoa mix with red fruit...,3426,10,False,False,Cabernet Sauvignon,0.212388,False,False
365,4105,Grenache,Grenache,1000-1999,test,US,California,Jim Gordon,"This medium-bodied wine walks its own path, sm...","This medium-bodied wine walks its own path, sm...",4104,10,False,False,Viognier,0.360399,False,False
412,4518,Garnacha,Grenache,1000-1999,test,Spain,Northern Spain,Michael Schachner,This barrel-fermented rosé is flat and smells ...,This barrel-fermented rosé is flat and smells ...,4517,10,False,False,Sauvignon Blanc,0.284305,False,False


In [7]:
bottom_df[bottom_df["grape"] == "Cabernet Franc"].head(5)

,review_id,variety,grape,bucket,split,country,province,taster_name,description,description_masked,emb_row,y,own_name_in_text,any_name_in_text,pred,confidence,correct,name_in_text
18,200,Cabernet Franc,Cabernet Franc,1000-1999,test,US,California,Jim Gordon,"This is a beatifully balanced, not-too-full-bo...","This is a beatifully balanced, not-too-full-bo...",199,2,False,False,Cabernet Sauvignon,0.238220,False,False
101,1090,Cabernet Franc,Cabernet Franc,1000-1999,test,US,California,Virginie Boone,"This is an impressively well-crafted, and beau...","This is an impressively well-crafted, and beau...",1089,2,False,True,Cabernet Sauvignon,0.546844,False,True
131,1402,Cabernet Franc,Cabernet Franc,1000-1999,test,US,Michigan,Alexander Peartree,"With 26 months in small oak barrels, aromas of...","With 26 months in small oak barrels, aromas of...",1401,2,False,False,Tempranillo,0.177142,False,False
186,2102,Cabernet Franc,Cabernet Franc,1000-1999,test,US,California,NaN,A very easy wine to like. It's so softly balan...,A very easy wine to like. It's so softly balan...,2101,2,False,False,Merlot,0.164713,False,False
238,2663,Cabernet Franc,Cabernet Franc,1000-1999,test,US,California,NaN,This Cab Franc is rich and ripe in red and bla...,This varietyname is rich and ripe in red and b...,2662,2,False,True,Cabernet Franc,0.606869,True,True



# 1. This first section will look at the predictions that were wrong and count how many times our model confused the identity of the grape

In [8]:
# This code is to see how many out of all of the bottom 3 reviews were wrong

class_idx = {g: i for i, g in enumerate(classes)} # grabs all of the grape types 

bottom_df["true_prob"] = proba_test_ens[
    bottom_df.index.to_numpy(),
    bottom_df["grape"].map(class_idx).to_numpy()
]

wrong = bottom_df[~bottom_df["correct"]].copy()
print(f"{len(wrong)} wrong out of {len(bottom_df)} bottom-3 reviews")


346 wrong out of 558 bottom-3 reviews


In [9]:
confusion_map = (
    wrong.groupby(["grape", "pred"])
         .agg(n_wrong=("pred", "size"),
              avg_conf=("confidence", "mean"),
              avg_true_prob=("true_prob", "mean"))
         .reset_index()
)

grape_totals = bottom_df["grape"].value_counts()
confusion_map["pct_of_grape"] = confusion_map["n_wrong"] / confusion_map["grape"].map(grape_totals)

confusion_map = confusion_map.sort_values(["grape", "n_wrong"], ascending=[True, False])
top_confusions = confusion_map.groupby("grape").head(5)
top_confusions.round(3)


,grape,pred,n_wrong,avg_conf,avg_true_prob,pct_of_grape
1,Cabernet Franc,Cabernet Sauvignon,51,0.433,0.110,0.203
10,Cabernet Franc,Pinot Noir,27,0.401,0.085,0.108
9,Cabernet Franc,Merlot,19,0.351,0.116,0.076
14,Cabernet Franc,Syrah,12,0.358,0.118,0.048
11,Cabernet Franc,Portuguese Red,9,0.392,0.042,0.036
25,Grenache,Pinot Noir,32,0.508,0.082,0.169
29,Grenache,Tempranillo,24,0.386,0.091,0.127
28,Grenache,Syrah,23,0.480,0.065,0.122
21,Grenache,Malbec,8,0.370,0.150,0.042
18,Grenache,Cabernet Sauvignon,6,0.288,0.029,0.032


In [10]:
wrong["conf_band"] = pd.cut(
    wrong["confidence"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["unsure (<0.4)", "leaning (0.4-0.7)", "confident (0.7+)"],
    include_lowest=True,
)

pd.crosstab(wrong["grape"], wrong["conf_band"], normalize="index").round(3)


conf_band,unsure (<0.4),leaning (0.4-0.7),confident (0.7+)
grape,,,
Cabernet Franc,0.563,0.377,0.060
Grenache,0.563,0.311,0.126
Pinot Blanc,0.539,0.316,0.145


# 2. Understaning words phrases that caused possible confusion

| Group | What it is | Why |
|-------|------------|-----|
| mistakes | A reviews the model called B | The confusing reviews |
| true_A | Normal A reviews (training set) | How A is usually described |
| true_B | Normal B reviews (training set) | How B is usually described |

Main Point:
We are pulling phrases & words that are used to describe A that sound like that they are describing B. For our presentation and write-up, possibly allowing us to find specific description in the dataset to show example to our audience to elaborate our idea of what potentially is causing our model to mix up the classification of the grape.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

train = model_df[model_df["split"] == "train"]
stop_words = list(ENGLISH_STOP_WORDS) + ["varietyname", "wine"]


In [12]:
def compare_pair(true_g, pred_g, min_conf=0.4, min_hits=3, top=20):
    mistakes = wrong[(wrong["grape"] == true_g) &
                     (wrong["pred"] == pred_g) &
                     (wrong["confidence"] >= min_conf)]["description_masked"]

    groups = {
        "mistakes": mistakes,
        "true_A": train.loc[train["grape"] == true_g, "description_masked"],
        "true_B": train.loc[train["grape"] == pred_g, "description_masked"],
    }
    print(f"{true_g} -> {pred_g}: {len(mistakes)} mistakes (confidence >= {min_conf}), "
          f"{len(groups['true_A'])} {true_g} train, {len(groups['true_B'])} {pred_g} train")